# Siemens Advanta- Bussines Case Project 2025/2026

**Project developed by Group V**:
   - Alano Gonçalves (20250457)
   - Catarina Martins (20221914)
   - João Carichas (20250507)
   - Marta Ribeiro (20221886)
   - Nicole Nogueira(20221961)

## 1. Imports

## 1.1 Import Libraries

In [7]:
import numpy as np
import pandas as pd
import zipfile
import xml.etree.ElementTree as ET

## 1.2 Import Datasets

In [8]:
def read_sales_sheet(zip_path, sheet_xml):
    with zipfile.ZipFile(zip_path) as z:
        with z.open('xl/sharedStrings.xml') as f:
            ss_root = ET.parse(f).getroot()
        ns = {'ns': 'http://purl.oclc.org/ooxml/spreadsheetml/main'}
        strings = []
        for si in ss_root.findall('.//ns:si', ns):
            texts = [t.text for t in si.findall('.//ns:t', ns) if t.text]
            strings.append(''.join(texts))

        with z.open(sheet_xml) as f:
            sheet_root = ET.parse(f).getroot()
        rows = []
        for row in sheet_root.findall('.//ns:row', ns):
            row_data = []
            for cell in row.findall('ns:c', ns):
                v = cell.find('ns:v', ns)
                t = cell.get('t')
                if v is not None:
                    row_data.append(strings[int(v.text)] if t == 's' else v.text)
                else:
                    row_data.append(None)
            rows.append(row_data)

    max_len = max(len(r) for r in rows)
    rows = [r + [None] * (max_len - len(r)) for r in rows]
    return pd.DataFrame(rows[1:], columns=rows[0])

In [9]:
SALES_PATH  = 'Case2_data_extract_share.xlsx'
MARKET_PATH = 'Case2_market_data_share.xlsx'
train_data = read_sales_sheet(SALES_PATH, 'xl/worksheets/sheet1.xml')
test_data = read_sales_sheet(SALES_PATH, 'xl/worksheets/sheet3.xml')

In [10]:
df_market = pd.read_excel(MARKET_PATH, sheet_name='Sheet1')
df_period_map = pd.read_excel(MARKET_PATH, sheet_name='Sheet2')
#df_period_map['Period'] = df_period_map['Period'].astype(int)

## 2. Data Exploration

## 2.1 Data Overview

In [11]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4237 entries, 0 to 4236
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Anon Period              4237 non-null   object
 1   TGL Biz Desc             4237 non-null   object
 2   TGL Business Unit        4237 non-null   object
 3   TGL Business Segment     4237 non-null   object
 4   TGL Business Subsegment  4237 non-null   object
 5   Orders cons. (anon)      4237 non-null   object
 6   Revenue cons. (anon)     4237 non-null   object
dtypes: object(7)
memory usage: 231.8+ KB


In [14]:
#changing the name of the variables do it is easy to track them
train_data = train_data.rename(columns={
    'Anon Period': 'Period',
    'TGL Biz Desc': 'Biz_Desc',
    'TGL Business Unit': 'Business_Unit',
    'TGL Business Segment': 'Segment',
    'TGL Business Subsegment': 'Subsegment',
    'Orders cons. (anon)': 'Orders',
    'Revenue cons. (anon)': 'Revenue'
})

In [17]:
#changing the datatypes so it is possible to visualize the data
train_data['Period'] = train_data['Period'].round().astype('Int32')
train_data['Orders'] = train_data['Orders'].round().astype('Int32')
train_data['Revenue'] = train_data['Revenue'].round().astype('Int32')

In [18]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4237 entries, 0 to 4236
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Period         4237 non-null   Int32 
 1   Biz_Desc       4237 non-null   object
 2   Business_Unit  4237 non-null   object
 3   Segment        4237 non-null   object
 4   Subsegment     4237 non-null   object
 5   Orders         4237 non-null   Int32 
 6   Revenue        4237 non-null   Int32 
dtypes: Int32(3), object(4)
memory usage: 194.6+ KB


In [26]:
df_market.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 78 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Period                                    180 non-null    int64  
 1   China_Core_Inflation_Rate                 178 non-null    float64
 2   China_Exports                             178 non-null    float64
 3   China_GDP                                 15 non-null     float64
 4   China_GDP_from_Construction               60 non-null     float64
 5   China_GDP_from_Manufacturing              60 non-null     float64
 6   China_Industrial_Production               176 non-null    float64
 7   China_Industrial_Production_Mom           170 non-null    float64
 8   China_Inflation_Rate                      178 non-null    float64
 9   China_Interest_Rate                       137 non-null    float64
 10  China_Steel_Production                

In [30]:
train_data.describe(include='all').round(2)

,Period,Biz_Desc,Business_Unit,Segment,Subsegment,Orders,Revenue
count,4237.0,4237,4237,4237,4237,4237.0,4237.0
unique,<NA>,1,4,24,134,<NA>,<NA>
top,<NA>,SSI,SSI027,SSI02784,SSI02780,<NA>,<NA>
freq,<NA>,4237,2243,1201,42,<NA>,<NA>
mean,22.51,NaN,NaN,NaN,NaN,40076623.58,35976412.57
std,12.24,NaN,NaN,NaN,NaN,70414862.66,63786043.94
min,1.0,NaN,NaN,NaN,NaN,-56101878.0,-12010755.0
25%,12.0,NaN,NaN,NaN,NaN,871389.0,1020192.0
50%,23.0,NaN,NaN,NaN,NaN,9281978.0,8850535.0
75%,33.0,NaN,NaN,NaN,NaN,43312847.0,36460128.0


In [31]:
df_market.describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
Period,180.0,-41.50,52.11,-131.00,-86.25,-41.50,3.25,48.00
China_Core_Inflation_Rate,178.0,1.37,0.60,-0.30,0.90,1.50,1.80,2.50
China_Exports,178.0,215.89,56.05,80.38,177.79,204.65,263.34,339.66
China_GDP,15.0,13031.13,4130.32,6192.56,10208.83,12537.56,16599.06,18743.80
China_GDP_from_Construction,60.0,33122.37,22602.81,4749.00,13793.25,28693.25,47756.28,88862.80
...,...,...,...,...,...,...,...,...
United_States_Industrial_Production,178.0,1.03,3.86,-17.20,-0.98,1.60,3.18,16.10
United_States_Industrial_Production_Mom,178.0,0.07,1.33,-13.20,-0.28,0.10,0.50,6.60
United_States_Inflation_Rate,178.0,2.60,1.98,-0.20,1.42,2.10,3.08,9.10
United_States_Interest_Rate,175.0,1.43,1.75,0.25,0.25,0.25,2.00,5.50


- The dataset has 42 months.
- There are rows in *Orders* and *Revenue* that are negative- Is it possible?

### 3.2. Checking Duplicates

In [32]:
#checking number of duplicates
train_data.duplicated().sum()

np.int64(0)

In [33]:
#checking number of duplicates
df_market.duplicated().sum()

np.int64(0)

- None of the datasets present duplicates.

### 3.3. Checking Missing Values

In [34]:
train_data.isna().sum()

Period           0
Biz_Desc         0
Business_Unit    0
Segment          0
Subsegment       0
Orders           0
Revenue          0
dtype: int64

In [39]:
df_market.isna().sum()

Period                                       0
China_Core_Inflation_Rate                    2
China_Exports                                2
China_GDP                                  165
China_GDP_from_Construction                120
                                          ... 
United_States_Industrial_Production          2
United_States_Industrial_Production_Mom      2
United_States_Inflation_Rate                 2
United_States_Interest_Rate                  5
United_States_Steel_Production               0
Length: 78, dtype: int64

In [40]:
pd.set_option('display.max_rows', None)
df_market.isna().sum()/len(df_market) * 100

Period                                       0.000000
China_Core_Inflation_Rate                    1.111111
China_Exports                                1.111111
China_GDP                                   91.666667
China_GDP_from_Construction                 66.666667
China_GDP_from_Manufacturing                66.666667
China_Industrial_Production                  2.222222
China_Industrial_Production_Mom              5.555556
China_Inflation_Rate                         1.111111
China_Interest_Rate                         23.888889
China_Steel_Production                       0.000000
France_Core_Inflation_Rate                   1.111111
France_Exports                               1.111111
France_GDP                                  91.666667
France_GDP_from_Construction                66.666667
France_GDP_from_Manufacturing               66.666667
France_Industrial_Production                 1.111111
France_Industrial_Production_Mom             1.111111
France_Inflation_Rate       